# Iris Classification using Mage AI

**Project:** Iris Flower Classification  
**Framework:** Mage AI  
**Model:** Decision Tree Classifier  
**Dataset:** Scikit-learn Iris dataset

## Objective
Load the Iris dataset, prepare training and testing data, train a Decision Tree model, evaluate predictions, validate data quality, and save predictions.

> This is a standalone companion reproduction of the Mage pipeline. The original Mage block files are included in `iris_project/`. Mage-specific decorators are omitted here so the notebook can run independently in Jupyter or Google Colab.

## 1. Import required libraries

In [1]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score
)

## 2. Load the Iris dataset
This reproduces the `load_iris` Mage data-loader block. The dataset contains 150 records, four input features, and one target column.

In [2]:
iris = load_iris(as_frame=True)
df = iris.frame

print("Dataset shape:", df.shape)
print("\nFirst five rows:")
display(df.head())
print("\nTarget class counts:")
print(df["target"].value_counts().sort_index())

Dataset shape: (150, 5)

First five rows:

Target class counts:
target
0    50
1    50
2    50
Name: count, dtype: int64


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


## 3. Prepare training and testing data
This reproduces `prepare_data.py`: 80% training data and 20% testing data, with `random_state=42` and stratification.

In [3]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("\nTraining class counts:")
print(y_train.value_counts().sort_index())
print("\nTesting class counts:")
print(y_test.value_counts().sort_index())

Training rows: 120
Testing rows: 30

Training class counts:
target
0    40
1    40
2    40
Name: count, dtype: int64

Testing class counts:
target
0    10
1    10
2    10
Name: count, dtype: int64


## 4. Train the Decision Tree model
This reproduces `train_model.py`. The copied Mage code uses `max_depth=5` by default and `random_state=42`. Its quality test requires accuracy of at least 70%.

In [4]:
max_depth = 5

model = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test).tolist()
test_accuracy = float(accuracy_score(y_test, predictions))

print("Model: DecisionTreeClassifier")
print("max_depth:", max_depth)
print(f"Test accuracy: {test_accuracy:.2%}")
print("Quality check (accuracy >= 70%):", test_accuracy >= 0.70)
assert test_accuracy >= 0.70, "Model accuracy is below 70%." 

Model: DecisionTreeClassifier
max_depth: 5
Test accuracy: 93.33%
Quality check (accuracy >= 70%): True


## 5. Experiment with different max_depth values
Compare the five depth values tested during the Mage work. A depth of 1 is below the 70% threshold; the other tested values pass.

In [5]:
experiment_results = []

for depth in [1, 2, 3, 4, 5]:
    experiment_model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    experiment_model.fit(X_train, y_train)
    experiment_predictions = experiment_model.predict(X_test)
    score = accuracy_score(y_test, experiment_predictions)
    experiment_results.append({
        "max_depth": depth,
        "accuracy": round(score * 100, 2),
        "quality_check": "Passed" if score >= 0.70 else "Failed"
    })

experiment_df = pd.DataFrame(experiment_results)
display(experiment_df)

,max_depth,accuracy,quality_check
0,1,66.67,Failed
1,2,93.33,Passed
2,3,96.67,Passed
3,4,93.33,Passed
4,5,93.33,Passed


## 6. Evaluate the model
This reproduces the `evaluate_model.py` block using predictions from the default `max_depth=5` model.

In [6]:
labels = [0, 1, 2]
cm = confusion_matrix(y_test, predictions, labels=labels)

confusion_table = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1", "Actual 2"],
    columns=["Predicted 0", "Predicted 1", "Predicted 2"]
)

metrics_table = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Score": [
        accuracy_score(y_test, predictions),
        precision_score(y_test, predictions, average="macro", zero_division=0),
        recall_score(y_test, predictions, average="macro", zero_division=0),
        f1_score(y_test, predictions, average="macro", zero_division=0)
    ]
})

print("Confusion matrix:")
display(confusion_table)
print("Evaluation metrics:")
display(metrics_table)

Confusion matrix:
Evaluation metrics:


,Predicted 0,Predicted 1,Predicted 2
Actual 0,10,0,0
Actual 1,0,9,1
Actual 2,0,1,9


,Metric,Score
0,Accuracy,0.933333
1,Precision,0.933333
2,Recall,0.933333
3,F1 Score,0.933333


## 7. Validate the dataset
This reproduces `validate_iris.py`: check 150 rows, zero missing values, and exactly three target classes.

In [7]:
validation_result = {
    "rows": int(len(df)),
    "missing_values": int(df.isnull().sum().sum()),
    "number_of_classes": int(df["target"].nunique())
}

assert validation_result["rows"] == 150
assert validation_result["missing_values"] == 0
assert validation_result["number_of_classes"] == 3

print("Validation results:")
display(pd.DataFrame([validation_result]))

Validation results:


,rows,missing_values,number_of_classes
0,150,0,3


## 8. Save prediction results
This reproduces `save_results.py`: save the actual and predicted class for each test record.

In [8]:
results = pd.DataFrame({
    "actual": y_test.reset_index(drop=True),
    "predicted": pd.Series(predictions)
})

results.to_csv("iris_predictions.csv", index=False)
print("Saved iris_predictions.csv")
print("Number of prediction rows:", len(results))
display(results.head(10))

Saved iris_predictions.csv
Number of prediction rows: 30


,actual,predicted
0,0,0
1,2,2
2,1,1
3,1,1
4,0,0
5,1,1
6,0,0
7,0,0
8,2,2
9,1,1


## 9. Conclusion

- Loaded the 150-row Iris dataset.
- Split it into 120 training rows and 30 testing rows.
- Trained a Decision Tree Classifier using `max_depth=5` and `random_state=42`.
- Checked model accuracy against the 70% quality threshold.
- Evaluated predictions with a confusion matrix, accuracy, precision, recall, and F1 score.
- Validated the dataset for missing values and the expected three classes.
- Saved 30 test predictions to `iris_predictions.csv`.

This notebook is a standalone reproduction of the logic in the Mage pipeline source files.